# Free Science Practice Notebook
## Line fitting, Pandas, calculus, uncertainty, gradients, and spectra

Core workflow:

$$
\boxed{\text{question}\to\text{data}\to\text{plot}\to\text{fit}\to\text{uncertainty}\to\text{explanation}}
$$


In [ ]:
import sympy as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sp.init_printing()


## 1. Linear fit

Model:

\[
y=mx+b
\]


In [ ]:
rng = np.random.default_rng(12)
x = np.linspace(0, 10, 25)
y = 2.4*x + 1.5 + rng.normal(0, 1.2, size=x.size)

df = pd.DataFrame({"x": x, "y_measured": y})
m_fit, b_fit = np.polyfit(x, y, 1)
df["y_fit"] = m_fit*x + b_fit
df["residual"] = df["y_measured"] - df["y_fit"]

print(f"slope = {m_fit:.4f}")
print(f"intercept = {b_fit:.4f}")
df.head()


In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(df["x"], df["y_measured"], label="data")
plt.plot(df["x"], df["y_fit"], label="fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear fit")
plt.grid()
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
plt.axhline(0)
plt.scatter(df["x"], df["residual"])
plt.xlabel("x")
plt.ylabel("residual")
plt.title("Residuals")
plt.grid()
plt.show()


## 2. Statistics

\[
\bar x=\frac1N\sum_i x_i,
\qquad
s=\sqrt{\frac{1}{N-1}\sum_i(x_i-\bar x)^2}
\]


In [ ]:
measurements = np.array([2.48,2.51,2.49,2.54,2.50,2.47,2.52,2.50,2.53,2.49])
print("mean =", measurements.mean())
print("sample std =", measurements.std(ddof=1))


## 3. Propagation of errors

For \(z=f(x,y)\),

\[
\sigma_z\approx
\sqrt{
\left(\frac{\partial z}{\partial x}\sigma_x\right)^2+
\left(\frac{\partial z}{\partial y}\sigma_y\right)^2
}.
\]


In [ ]:
x_sym, y_sym = sp.symbols("x y", positive=True)
sx, sy = sp.symbols("sigma_x sigma_y", positive=True)

z = x_sym/y_sym
sigma_z = sp.sqrt(
    (sp.diff(z, x_sym)*sx)**2 +
    (sp.diff(z, y_sym)*sy)**2
)

display(sp.Eq(sp.Symbol("z"), z))
display(sp.Eq(sp.Symbol(r"\sigma_z"), sp.simplify(sigma_z)))


## 4. Gradient and electrostatics

For a scalar potential \(V(x,y)\),

\[
\nabla V=
\begin{bmatrix}
\partial V/\partial x\\
\partial V/\partial y
\end{bmatrix},
\qquad
\mathbf E=-\nabla V.
\]


In [ ]:
x_sym, y_sym = sp.symbols("x y", real=True)
V = x_sym**2 + 2*x_sym*y_sym + 3*y_sym**2

gradV = sp.Matrix([sp.diff(V,x_sym), sp.diff(V,y_sym)])
E = -gradV

display(V)
display(gradV)
display(E)


## 5. Line-integral check

For a conservative field,

\[
\int_A^B \nabla V\cdot d\mathbf r=V(B)-V(A).
\]


In [ ]:
t = sp.symbols("t", real=True)

x_t = t
y_t = 2*t

V_path = V.subs({x_sym:x_t, y_sym:y_t})
line_integral = sp.integrate(sp.diff(V_path,t), (t,0,1))

V_A = V.subs({x_sym:0, y_sym:0})
V_B = V.subs({x_sym:1, y_sym:2})

display(line_integral)
display(sp.simplify(V_B - V_A))


## 6. Exponential science model

\[
y(t)=Ae^{-kt}.
\]


In [ ]:
t_sym, A, k = sp.symbols("t A k", positive=True)
decay = A*sp.exp(-k*t_sym)

display(decay)
display(sp.diff(decay,t_sym))


In [ ]:
t_data = np.linspace(0, 5, 35)
rng = np.random.default_rng(5)

y_clean = 4.0*np.exp(-0.42*t_data)
y_noisy = np.clip(y_clean + rng.normal(0,0.04,t_data.size), 1e-8, None)

slope, intercept = np.polyfit(t_data, np.log(y_noisy), 1)

print("A estimate =", np.exp(intercept))
print("k estimate =", -slope)


## 7. Synthetic spectroscopy-style trace

This is synthetic teaching data, not experimental or medical data.


In [ ]:
wavelength_nm = np.linspace(500,700,500)
center_nm = 610
width_nm = 14

intensity = 1.0 - 0.65*np.exp(-0.5*((wavelength_nm-center_nm)/width_nm)**2)

spectrum_df = pd.DataFrame({
    "wavelength_nm": wavelength_nm,
    "intensity": intensity
})

spectrum_df.head()


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(spectrum_df["wavelength_nm"], spectrum_df["intensity"])
plt.xlabel("Wavelength [nm]")
plt.ylabel("Relative intensity")
plt.title("Synthetic spectral feature")
plt.grid()
plt.show()


## 8. Extract interpretable spectral features


In [ ]:
i_min = np.argmin(intensity)

features = pd.DataFrame({
    "feature": [
        "minimum intensity",
        "wavelength at minimum",
        "mean intensity",
        "integrated area"
    ],
    "value": [
        intensity[i_min],
        wavelength_nm[i_min],
        intensity.mean(),
        np.trapz(intensity, wavelength_nm)
    ]
})

features


## 9. Practice

1. Increase the noise and inspect residuals.
2. Replace the line with a quadratic fit.
3. Derive the gradient by hand before running SymPy.
4. Change the line-integral path but keep endpoints fixed.
5. Change the exponential \(k\) value.
6. Move the synthetic spectral feature.
7. Add noise to the spectrum.
8. Write five sentences explaining one plot.

The hand skill is:

\[
\boxed{\text{model}\to\text{numbers}\to\text{table}\to\text{plot}\to\text{fit}\to\text{uncertainty}}
\]
